# TP 4 — Graphes : parcours, plus court chemin et arbre couvrant minimal
### Algorithmique avancée et complexité — Master IA et Master RSD · Chapitre 3

| | |
|---|---|
| **Cours & TD** | Dr. Benyamina Ahmed |
| **TP** | Mahkouka Mohamed Fouad |
| **Durée** | 1 séance |

---
## 🎯 Objectif

Implémenter un **graphe non orienté pondéré** et tester :
- les parcours **en largeur (BFS)** et **en profondeur (DFS)** ;
- l'algorithme de **Dijkstra** (plus court chemin) ;
- l'algorithme de **Kruskal** (arbre couvrant minimal, ACM) ;
- le **temps d'exécution** de chaque opération.

> **Rappel :** un graphe non orienté est un ensemble de sommets reliés par des arêtes (pondérées ou non). On le représente par une **liste d'adjacence** : pour chaque sommet, la liste de ses voisins avec le poids de l'arête.

**Contraintes :** Python 3 uniquement (pas de bibliothèque externe ; `collections` et `heapq` font partie de Python), code commenté, résultats clairement affichés.

---
# Outils

**Exécutez cette cellule en premier.** `verifier` affiche ✅ ou ❌ ; `chrono_ms(f, *args)` donne le temps moyen en **millisecondes** sur 5 exécutions.

In [ ]:
def verifier(nom, obtenu, attendu):
    print(("✅" if obtenu == attendu else "❌"), nom, "| obtenu :", obtenu, "| attendu :", attendu)


import heapq
import random
import sys
import time
from collections import deque
from statistics import linear_regression
import math


def chrono_ms(f, *args, repetitions=5):
    # Temps moyen de f(*args) en millisecondes, sur 5 exécutions.
    total = 0.0
    for _ in range(repetitions):
        start = time.perf_counter()
        f(*args)
        total += time.perf_counter() - start
    return total / repetitions * 1000


print("Outils prêts ✅")

---
# Partie 1 — Le graphe étudié *(fourni)*

Graphe non orienté pondéré à 7 sommets et 11 arêtes ; source des parcours et de Dijkstra : **A**.

| Sommet | Voisins (poids) |
|---|---|
| A | B (4), D (8), E (1) |
| B | A (4), C (2), E (3) |
| C | B (2), E (8), F (2) |
| D | A (8), E (10) |
| E | A (1), B (3), C (8), D (10), F (7), G (6) |
| F | C (2), E (7), G (4) |
| G | E (6), F (4) |

La classe `Graph` stocke la liste d'adjacence dans un dictionnaire. Lisez-la et exécutez la cellule.

In [ ]:
class Graph:
    def __init__(self):
        self.adj = {}           # sommet → liste de (voisin, poids)

    def add_vertex(self, s):
        if s not in self.adj:
            self.adj[s] = []

    def add_edge(self, u, v, w):
        self.add_vertex(u)
        self.add_vertex(v)
        self.adj[u].append((v, w))
        self.adj[v].append((u, w))      # non orienté : dans les deux sens

    def vertices(self):
        return list(self.adj.keys())

    def edges(self):
        # chaque arête une seule fois, sous la forme (u, v, w)
        seen, res = set(), []
        for u in self.adj:
            for v, w in self.adj[u]:
                if (v, u) not in seen:
                    seen.add((u, v))
                    res.append((u, v, w))
        return res


g = Graph()
for u, v, w in [("A", "B", 4), ("A", "D", 8), ("A", "E", 1), ("B", "C", 2), ("B", "E", 3), ("C", "E", 8),
                ("C", "F", 2), ("D", "E", 10), ("E", "F", 7), ("E", "G", 6), ("F", "G", 4)]:
    g.add_edge(u, v, w)

for s in g.vertices():
    print(s, ":", g.adj[s])
verifier("nombre d'arêtes", len(g.edges()), 11)

---
# Partie 2 — Parcours BFS et DFS

**BFS (largeur)** : on visite la source, puis tous ses voisins, puis les voisins des voisins… avec une **file** (`deque`) : `popleft()` pour défiler, `append()` pour enfiler. Le **niveau** d'un sommet est sa distance en nombre d'arêtes depuis A.

**DFS (profondeur)** : on part d'un sommet et on va le plus loin possible avant de revenir en arrière. Version **récursive** : marquer `u` comme vu, l'ajouter à l'ordre de visite, puis visiter chaque voisin pas encore vu.

Les voisins sont toujours examinés **dans l'ordre de la liste d'adjacence**.

In [ ]:
def bfs(graph, source):
    # Renvoie (ordre de visite, niveaux) ; niveau = distance en nombre d'arêtes depuis source.
    visited = [source]
    levels = {source: 0}
    queue = deque([source])
    # À COMPLÉTER : tant que la file n'est pas vide, défiler u et enfiler ses voisins pas encore vus
    return visited, levels


def dfs(graph, source):
    # Parcours en profondeur récursif ; renvoie l'ordre de visite.
    visited = []
    seen = set()

    def visit(u):
        # À COMPLÉTER
        pass

    visit(source)
    return visited

In [ ]:
order, levels = bfs(g, "A")
verifier("BFS ordre", order, ["A", "B", "D", "E", "C", "F", "G"])
verifier("BFS niveaux", levels, {"A": 0, "B": 1, "D": 1, "E": 1, "C": 2, "F": 2, "G": 2})
verifier("DFS ordre", dfs(g, "A"), ["A", "B", "C", "E", "D", "F", "G"])

> ✍️ **Pourquoi E est-il au niveau 1 en BFS, mais visité en 4e position en DFS ?** …

---
# Partie 3 — Plus court chemin : Dijkstra

On garde pour chaque sommet la meilleure distance connue `dist` et son prédécesseur `pred`. On extrait toujours le sommet de **plus petite distance** grâce à une file de priorité (`heapq`), puis on essaie d'améliorer ses voisins :

```
si dist[u] + w < dist[v] :
    dist[v] ← dist[u] + w ; pred[v] ← u ; ajouter (dist[v], v) dans la file
```

✍️ Complétez la boucle de `dijkstra`, puis `path(pred, cible)` qui reconstruit le chemin en remontant les prédécesseurs (puis en inversant la liste).

In [ ]:
def dijkstra(graph, source):
    # Renvoie (dist, pred) : distance minimale et prédécesseur de chaque sommet.
    dist = {s: float("inf") for s in graph.adj}
    pred = {s: None for s in graph.adj}
    dist[source] = 0
    pq = [(0, source)]                  # file de priorité (distance, sommet)
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue                    # entrée périmée
        # À COMPLÉTER : pour chaque voisin v, si d + w < dist[v] : mettre à jour dist, pred, et pousser dans pq
    return dist, pred


def path(pred, target):
    # Remonte les prédécesseurs depuis target jusqu'à la source.
    res = []
    # À COMPLÉTER
    return res

In [ ]:
dist, pred = dijkstra(g, "A")
verifier("distances depuis A", dist, {"A": 0, "B": 4, "D": 8, "E": 1, "C": 6, "F": 8, "G": 7})
verifier("chemin A → F", path(pred, "F"), ["A", "E", "F"])
verifier("chemin A → C", path(pred, "C"), ["A", "B", "C"])

for s in g.vertices():
    print(f"A → {s} : {' → '.join(path(pred, s))}  (coût {dist[s]})")

> ✍️ **Vers F, deux chemins coûtent 8 (A → E → F et A → B → C → F). Pourquoi Dijkstra garde-t-il A → E → F ?** …

---
# Partie 4 — Arbre couvrant minimal : Kruskal

Un **arbre couvrant minimal (ACM)** relie **tous** les sommets avec $V - 1$ arêtes, sans cycle, pour un poids total minimal.

**Kruskal** : trier les arêtes par poids croissant ; prendre chaque arête **si elle ne crée pas de cycle**. Pour détecter un cycle, la structure `UnionFind` (fournie) regroupe les sommets déjà reliés : `uf.union(u, v)` renvoie `False` si `u` et `v` sont déjà dans le même groupe.

In [ ]:
class UnionFind:
    def __init__(self, vertices):
        self.parent = {s: s for s in vertices}
        self.rank = {s: 0 for s in vertices}

    def find(self, x):                  # représentant de l'ensemble de x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):              # False si x et y sont déjà reliés (cycle)
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True

In [ ]:
def kruskal(graph):
    # Renvoie (arêtes de l'ACM, poids total).
    edges = sorted(graph.edges(), key=lambda e: e[2])      # poids croissant
    uf = UnionFind(graph.vertices())
    mst, total = [], 0
    # À COMPLÉTER : garder chaque arête (u, v, w) qui ne crée pas de cycle (uf.union renvoie True)
    return mst, total

In [ ]:
mst, total = kruskal(g)
for u, v, w in mst:
    print(f"{u} – {v} ({w})")
verifier("poids total de l'ACM", total, 20)
verifier("nombre d'arêtes = V − 1", len(mst), len(g.vertices()) - 1)
verifier("D est relié par A – D (8)", ("A", "D", 8) in mst, True)

> ✍️ **Pourquoi l'arête E – G (6) est-elle rejetée ?** …

---
# Partie 5 — Temps d'exécution

## 5.1 Sur le graphe étudié

In [ ]:
print(f"{'Algorithme':<10} {'Complexité théorique':<24} {'Temps moyen (ms)':>18}")
for name, cplx, f in [("BFS", "O(V + E)", lambda: bfs(g, "A")), ("DFS", "O(V + E)", lambda: dfs(g, "A")),
                      ("Dijkstra", "O((V + E) log V)", lambda: dijkstra(g, "A")), ("Kruskal", "O(E log E)", lambda: kruskal(g))]:
    print(f"{name:<10} {cplx:<24} {chrono_ms(f):>18.4f}")

## 5.2 Sur des graphes de plus en plus grands
On génère des graphes aléatoires **connexes** avec $V$ sommets et environ $E = 3V$ arêtes, et on mesure chaque algorithme. *(Exécutez seulement.)*

In [ ]:
sys.setrecursionlimit(50_000)       # le DFS récursif peut descendre très profond

def random_graph(V, seed=0):
    rnd = random.Random(seed)
    G = Graph()
    for i in range(1, V):                                   # chaîne : garantit la connexité
        G.add_edge(i - 1, i, rnd.randint(1, 100))
    for _ in range(2 * V):                                  # arêtes en plus, au hasard
        u, v = rnd.randrange(V), rnd.randrange(V)
        if u != v:
            G.add_edge(u, v, rnd.randint(1, 100))
    return G

sizes = [100, 1_000, 5_000, 20_000]
times = {"BFS": [], "DFS": [], "Dijkstra": [], "Kruskal": []}
print(f"{'V':>7} {'E':>7} | " + " ".join(f"{k:>10}" for k in times) + "   (ms)")
for V in sizes:
    G = random_graph(V)
    for name, f in [("BFS", lambda: bfs(G, 0)), ("DFS", lambda: dfs(G, 0)),
                    ("Dijkstra", lambda: dijkstra(G, 0)), ("Kruskal", lambda: kruskal(G))]:
        times[name].append(chrono_ms(f, repetitions=3))
    print(f"{V:>7,} {len(G.edges()):>7,} | " + " ".join(f"{times[k][-1]:>10.2f}" for k in times))

for name, values in times.items():
    pts = [(n, v) for n, v in zip(sizes, values) if v > 0]
    if len(pts) > 1:
        slope = linear_regression([math.log10(n) for n, _ in pts], [math.log10(v) for _, v in pts]).slope
        print(f"{name:<9} pente log-log ≈ {slope:.2f}")

> ✍️ **Les pentes sont-elles conformes aux complexités théoriques ?** …

---
# 📤 Travail à rendre

1. Ce notebook complété : **Restart & Run All** sans erreur, toutes les vérifications ✅.
2. Les lignes ✍️ complétées.
3. Envoyez sur GitHub :

```bash
git add tp4_graphes/
git commit -m "TP4 : graphes"
git push
```

## 🎯 À retenir

| Algorithme | Rôle | Structure clé | Complexité |
|---|---|---|---|
| BFS | parcours par niveaux | file (`deque`) | $O(V + E)$ |
| DFS | parcours en profondeur | récursion (pile) | $O(V + E)$ |
| Dijkstra | plus courts chemins (poids ≥ 0) | file de priorité (`heapq`) | $O((V + E) \log V)$ |
| Kruskal | arbre couvrant minimal | tri + Union-Find | $O(E \log E)$ |

- Dijkstra ne fonctionne qu'avec des **poids positifs**.
- Un ACM de $V$ sommets a toujours $V - 1$ arêtes.